In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/09/14 13:57:02 WARN Utils: Your hostname, GANIU-ODEYINKA resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/14 13:57:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/14 13:57:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [6]:
df_green.createOrReplaceTempView('green')

In [ ]:
#GROUPBY

In [19]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS no_of_records
    
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
--ORDER BY
  --  1, 2
""")

In [20]:
df_green_revenue.show()

[Stage 41:==================================================>       (7 + 1) / 8]

+-------------------+----+------------------+-------------+
|               hour|zone|            amount|no_of_records|
+-------------------+----+------------------+-------------+
|2020-01-22 12:00:00|  74|1000.1799999999988|           65|
|2020-01-04 20:00:00|  69|              11.8|            1|
|2020-01-06 15:00:00| 165|            165.19|            8|
|2020-01-23 16:00:00| 179|             102.2|            4|
|2020-01-10 20:00:00|  66|            405.88|           21|
|2020-01-10 10:00:00| 166| 498.1300000000001|           29|
|2020-01-10 09:00:00| 168|            162.76|            9|
|2020-01-11 20:00:00|  41| 741.7199999999997|           56|
|2020-01-08 18:00:00| 254|             96.64|            6|
|2020-01-15 13:00:00| 181|            175.56|           10|
|2020-01-03 14:00:00|  74| 770.6899999999998|           57|
|2020-01-01 01:00:00|  65|246.86000000000004|           16|
|2020-01-08 09:00:00|  33|            625.03|           36|
|2020-01-22 15:00:00| 150|             8

In [21]:
df_green_revenue.write.parquet('data/report/revenue/green', mode='overwrite')

25/09/14 14:26:09 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [22]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [23]:
df_yellow.createOrReplaceTempView('yellow')

In [24]:
df_yellow.createOrReplaceTempView('yellow')

In [26]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS no_of_records
    
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [27]:
df_yellow_revenue.show()

[Stage 48:==============================================>         (10 + 2) / 12]

+-------------------+----+------------------+-------------+
|               hour|zone|            amount|no_of_records|
+-------------------+----+------------------+-------------+
|2020-01-15 10:00:00|  68| 3982.539999999996|          226|
|2020-01-20 17:00:00| 113| 2187.419999999999|          157|
|2020-01-05 13:00:00|  68| 3606.959999999996|          213|
|2020-01-09 19:00:00| 262|2204.9999999999986|          140|
|2020-01-04 09:00:00| 230| 3695.689999999998|          210|
|2020-01-26 15:00:00| 236| 8049.910000000014|          553|
|2020-01-15 22:00:00| 239| 2512.369999999999|          158|
|2020-01-03 19:00:00| 186|  8718.91000000002|          489|
|2020-01-01 00:00:00| 114| 6256.430000000005|          333|
|2020-01-07 00:00:00| 158|            258.08|           13|
|2020-01-31 19:00:00| 237| 9720.220000000027|          636|
|2020-01-31 10:00:00| 143|2686.4299999999976|          184|
|2020-01-23 19:00:00| 137| 2560.579999999998|          152|
|2020-01-19 18:00:00| 211|2447.769999999

In [32]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

25/09/14 15:29:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/09/14 15:29:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [68]:
#JOIN - two large tables (supposed) - (spark does this using RESHUFFLING method)

In [48]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('no_of_records', 'green_no_of_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('no_of_records', 'yellow_no_of_records')

In [49]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [50]:
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_no_of_records: bigint, yellow_amount: double, yellow_no_of_records: bigint]

In [ ]:
df_join.show()

In [51]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

25/09/14 15:58:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [52]:
#JOINS - one large and one small table (spark does this using BROADCASTING method)

In [53]:
#to read dataset using spark
df_join = spark.read.parquet('data/report/revenue/total')

In [56]:
df_zones = spark.read.parquet('zones/')

In [58]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [59]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [67]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue_zones')

25/09/14 16:32:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                